# The Causal Inference Problem: Why Naive Regression Fails

**Goal:** See why OLS fails for causal inference with many controls, and preview how DML fixes it.

**Time:** 15 minutes

In this notebook, you will simulate an OPEC production cut scenario where confounders bias naive OLS,
watch OLS standard errors explode as you add controls, and see DML recover the true treatment effect
even with 200 controls.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## The Setup: OPEC Production Cuts and Calendar Spreads

We simulate a commodity market where:
- **Treatment D:** OPEC production cut size (million bbl/day)
- **Outcome Y:** WTI 1-3 month calendar spread
- **Controls X:** Global demand indicators, inventory levels, shipping rates, etc.
- **True causal effect:** theta = 2.0 (each 1M bbl/day cut raises spread by 2.0)

The key confounder is global demand -- it causes OPEC to cut (they cut when demand is falling)
AND directly affects spreads (falling demand tightens the spread).

In [ ]:
# Data generating process with known ground truth
n = 1000
true_effect = 2.0

# 5 observable confounders
X = np.random.randn(n, 5)
confounder_names = ['global_demand', 'inventories', 'shipping_rates',
                    'refinery_util', 'geopolitical_risk']

# Treatment: production cut correlated with demand and inventories
D = 0.5 * X[:, 0] + 0.3 * X[:, 1] + np.random.randn(n) * 0.5

# Outcome: spread depends on treatment AND confounders
Y = true_effect * D + 1.5 * X[:, 0] + 0.8 * X[:, 2] + np.random.randn(n)

print(f'Simulated {n} observations')
print(f'True causal effect: {true_effect}')
print(f'Correlation(D, demand): {np.corrcoef(D, X[:, 0])[0, 1]:.3f}')
print(f'Correlation(Y, demand): {np.corrcoef(Y, X[:, 0])[0, 1]:.3f}')

## Naive OLS: Omitting Controls

Run OLS regressing Y on D without any controls. The estimate will be biased
because global demand (X[0]) confounds the relationship.

In [ ]:
# Naive OLS: Y ~ D (no controls)
naive_model = sm.OLS(Y, sm.add_constant(D)).fit()

# OLS with correct controls: Y ~ D + X
DX = np.column_stack([D, X])
controlled_model = sm.OLS(Y, sm.add_constant(DX)).fit()

print('=' * 50)
print(f'True causal effect:       {true_effect:.2f}')
print(f'Naive OLS (no controls):  {naive_model.params[1]:.2f}  (SE: {naive_model.bse[1]:.3f})')
print(f'OLS with 5 controls:      {controlled_model.params[1]:.2f}  (SE: {controlled_model.bse[1]:.3f})')
print('=' * 50)
print(f'\nNaive bias: {naive_model.params[1] - true_effect:+.2f}')
print(f'Controlled bias: {controlled_model.params[1] - true_effect:+.2f}')
print(f'\nWith few controls, OLS with controls works well.')
print(f'But what happens with 200 controls?')

## Scaling Up: What Happens with 200 Controls

In real commodity markets, you have hundreds of potential controls: macroeconomic indicators,
inventory data across locations, weather, shipping routes, refinery status, etc.

We simulate p=200 controls (only 3 are truly relevant) and watch OLS standard errors explode.

In [ ]:
# Scale up to varying numbers of controls
p_values = [5, 20, 50, 100, 200]
results = []

for p in p_values:
    np.random.seed(42)
    X_p = np.random.randn(n, p)
    D_p = 0.5 * X_p[:, 0] + 0.3 * X_p[:, 1] + np.random.randn(n) * 0.5
    Y_p = true_effect * D_p + 1.5 * X_p[:, 0] + 0.8 * X_p[:, 2] + np.random.randn(n)

    DX_p = np.column_stack([D_p, X_p])
    model = sm.OLS(Y_p, sm.add_constant(DX_p)).fit()

    results.append({
        'p': p,
        'estimate': model.params[1],
        'se': model.bse[1],
        'ci_width': 2 * 1.96 * model.bse[1],
        'p_over_n': p / n
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f'\nTrue effect: {true_effect:.2f}')
print('\nAs p/n grows, standard errors and CI width increase dramatically.')

## Visualise the Variance Explosion

Plot the 95% confidence interval width as a function of the number of controls.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CI width vs number of controls
axes[0].plot(results_df['p'], results_df['ci_width'], 'o-',
             linewidth=2, markersize=8, color='crimson')
axes[0].set_xlabel('Number of Controls (p)', fontsize=12)
axes[0].set_ylabel('95% CI Width', fontsize=12)
axes[0].set_title('OLS Confidence Interval Explodes with Many Controls',
                   fontsize=13, fontweight='bold')
axes[0].axhline(y=0.5, color='green', linestyle='--', alpha=0.7,
                label='Useful precision')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: Estimates with error bars
axes[1].errorbar(results_df['p'], results_df['estimate'],
                 yerr=1.96 * results_df['se'],
                 fmt='o-', linewidth=2, markersize=8, capsize=5,
                 color='steelblue')
axes[1].axhline(y=true_effect, color='red', linestyle='--',
                linewidth=2, label=f'True effect = {true_effect}')
axes[1].set_xlabel('Number of Controls (p)', fontsize=12)
axes[1].set_ylabel('Treatment Effect Estimate', fontsize=12)
axes[1].set_title('OLS Point Estimates with 95% CI',
                   fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('The point estimate may be roughly correct, but the CI is so wide it is useless.')

## The Frisch-Waugh-Lovell Decomposition

FWL says: residualise both Y and D on X, then regress the residuals.
This gives the same estimate as OLS with controls, but reveals the key insight:
the treatment effect is about the **relationship between residuals**.

In [ ]:
# FWL decomposition with 5 controls
X5 = X[:, :5]

# Step 1: Residualise Y on X
resid_Y = sm.OLS(Y, sm.add_constant(X5)).fit().resid

# Step 2: Residualise D on X
resid_D = sm.OLS(D, sm.add_constant(X5)).fit().resid

# Step 3: Regress residuals
fwl_model = sm.OLS(resid_Y, sm.add_constant(resid_D)).fit()

print(f'FWL estimate:             {fwl_model.params[1]:.4f}')
print(f'OLS with controls:        {controlled_model.params[1]:.4f}')
print(f'Match: {np.isclose(fwl_model.params[1], controlled_model.params[1], atol=1e-6)}')
print(f'\nFWL reveals: the treatment effect is about residual correlations.')
print(f'DML replaces the OLS first stages with ML models.')

## DML Preview: ML Residualisation with 200 Controls

Replace OLS first stages with random forests. Use `cross_val_predict` for
out-of-sample predictions (a simplified version of cross-fitting).

In [ ]:
# Generate the high-dimensional dataset
np.random.seed(42)
p = 200
X_large = np.random.randn(n, p)
D_large = 0.5 * X_large[:, 0] + 0.3 * X_large[:, 1] + np.random.randn(n) * 0.5
Y_large = true_effect * D_large + 1.5 * X_large[:, 0] + 0.8 * X_large[:, 2] + np.random.randn(n)

# DML approach: ML residualisation
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)

# Cross-validated predictions (out-of-sample)
Y_hat = cross_val_predict(rf, X_large, Y_large, cv=5)
D_hat = cross_val_predict(rf, X_large, D_large, cv=5)

# Residuals
resid_Y_ml = Y_large - Y_hat
resid_D_ml = D_large - D_hat

# Final regression on residuals
dml_model = sm.OLS(resid_Y_ml, sm.add_constant(resid_D_ml)).fit()

# Compare all approaches
DX_large = np.column_stack([D_large, X_large])
ols_large = sm.OLS(Y_large, sm.add_constant(DX_large)).fit()
naive_large = sm.OLS(Y_large, sm.add_constant(D_large)).fit()

print('=' * 60)
print(f'{"Method":<30} {"Estimate":>8} {"SE":>8} {"CI Width":>10}')
print('=' * 60)
print(f'{"True effect":<30} {true_effect:>8.2f}')
print(f'{"Naive OLS (no controls)":<30} {naive_large.params[1]:>8.2f} {naive_large.bse[1]:>8.3f} {2*1.96*naive_large.bse[1]:>10.3f}')
print(f'{"OLS with 200 controls":<30} {ols_large.params[1]:>8.2f} {ols_large.bse[1]:>8.3f} {2*1.96*ols_large.bse[1]:>10.3f}')
print(f'{"DML (RF + cross-val)":<30} {dml_model.params[1]:>8.2f} {dml_model.bse[1]:>8.3f} {2*1.96*dml_model.bse[1]:>10.3f}')
print('=' * 60)

## Visualise: Residual Plots

Plot the residual relationship for both FWL (OLS first stages) and DML (ML first stages).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: FWL residuals (5 controls)
axes[0].scatter(resid_D, resid_Y, alpha=0.3, s=10, color='steelblue')
x_range = np.linspace(resid_D.min(), resid_D.max(), 100)
axes[0].plot(x_range, fwl_model.params[0] + fwl_model.params[1] * x_range,
             color='red', linewidth=2, label=f'slope = {fwl_model.params[1]:.2f}')
axes[0].set_xlabel('D residuals (treatment variation)', fontsize=11)
axes[0].set_ylabel('Y residuals (outcome variation)', fontsize=11)
axes[0].set_title('FWL: OLS Residuals (5 controls)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Right: DML residuals (200 controls)
axes[1].scatter(resid_D_ml, resid_Y_ml, alpha=0.3, s=10, color='forestgreen')
x_range_ml = np.linspace(resid_D_ml.min(), resid_D_ml.max(), 100)
axes[1].plot(x_range_ml, dml_model.params[0] + dml_model.params[1] * x_range_ml,
             color='red', linewidth=2, label=f'slope = {dml_model.params[1]:.2f}')
axes[1].set_xlabel('D residuals (treatment variation)', fontsize=11)
axes[1].set_ylabel('Y residuals (outcome variation)', fontsize=11)
axes[1].set_title('DML: ML Residuals (200 controls)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Both plots show the same causal relationship (slope near 2.0).')
print('DML achieves this even with 200 controls where OLS struggles.')

## Summary

**What you learned:**
1. Naive OLS without controls is biased due to omitted variable bias
2. OLS with many controls has exploding standard errors (variance-bias tradeoff)
3. Frisch-Waugh-Lovell decomposes the treatment effect into a residual-on-residual regression
4. DML replaces OLS first stages with ML, recovering the true effect even with 200 controls

**What is next:**
- **Module 01:** Why Lasso + OLS (the obvious fix) also fails for causal inference
- **Module 02:** The orthogonalisation trick -- Robinson's partially linear model
- **Module 03:** Neyman orthogonal scores -- why DML is robust to ML errors

**Key takeaway:** The treatment effect is about residuals. OLS can only compute linear residuals.
DML computes flexible ML residuals while maintaining valid statistical inference.